# Agentic RAG (Router + Retriever) — Colab runner

Runs the same `crew.py` logic used locally. Run the cells in order:
1. Install dependencies
2. Upload `crew.py` and the PDF (`transformer_research_paper-dataset.pdf`)
3. Set API keys (via Colab Secrets if configured, otherwise you'll be prompted)
4. Ask a question

**Colab Secrets setup (recommended, one-time):** click the key icon in the left sidebar, add `OPENAI_API_KEY` and `TAVILY_API_KEY`, and toggle "Notebook access" on for each. This avoids re-entering keys every session.

In [ ]:
# 1. Install dependencies (Colab's default runtime is Python 3.12, which has
# prebuilt wheels for everything here — no build-from-source issues).
!pip install -q "crewai[tools]" tavily-python

In [ ]:
# 2. Upload crew.py and the PDF. Select both files in the picker that appears.
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
# 3. Set API keys — tries Colab Secrets first, falls back to a manual
# (hidden-input) prompt if secrets aren't set up.
import os

try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    print("Loaded API keys from Colab Secrets.")
except Exception:
    from getpass import getpass

    print("Colab Secrets not found — enter keys manually (input is hidden):")
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
    os.environ["TAVILY_API_KEY"] = getpass("TAVILY_API_KEY: ")

In [ ]:
# 4. Build the crew (indexes the PDF into a local ChromaDB store — takes a
# little time on first run) and ask a question.
from crew import build_crew

crew = build_crew("transformer_research_paper-dataset.pdf")

In [ ]:
question = "What is the attention mechanism described in the paper?"  # try a PDF-scoped question...
# question = "What is the latest news in AI this week?"  # ...or a web-scoped one

result = crew.kickoff(inputs={"question": question})
print("\n=== Final Answer ===")
print(result)